### Regresión logística binaria (*benchmark*)

Se planteará inicialmente un modelo base de sencilla aplicación: **modelo de regresión logística binario**, teniendo en cuenta solo los efectos principales de cada variable. Posteriormente se comparará dicho modelo con otras opciones más complejas basadas en *boosting*

#### Librerías


In [11]:
import polars as pl
import pyprojroot
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline

#### Carga de datos

In [6]:
ROOT = pyprojroot.here()
datos_limpios = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio.parquet")

#### Preselección de variables

Vamos a utilizar como predictores a todas las variables definidas en el proceso de featuring, excluyendo las variables referidas a los id's, los nombres de los pitchers y batters y las variables plate_x y plate_z (ya que se encuentran representadas por relative_x y relative_height). 

In [16]:
numeric_features = [
    "release_speed",
    "balls",
    "strikes",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "sz_mid",
    "strike_zone_size",
    "movement_complexity",
    "relative_height",
    "relative_x",
    "pitch_location",
    "complex_speed",
]

binary_features = [
    "platoon_advantage",
    "is_strike_zone",
    "is_shadow_zone",
]

categorical_features = [
    "pitch_type",
    "stand",
    "p_throws",
]

X = datos_limpios.select(
    numeric_features +
    binary_features +
    categorical_features
)

y = datos_limpios["swing"]

Escalo los datos y hago las conversiones necesarias de las variables.

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
        (
            "bin",
            "passthrough",
            binary_features,
        ),
    ]
)

Modelo:

--aa--

In [ ]:
LogisticRegressionCV(
    penalty="l2",
    solver="lbfgs",
    scoring="neg_log_loss",
    cv=5,
    Cs=np.logspace(-3, 3, 10),
    max_iter=1000,
    n_jobs=-1
)

Pipeline:

In [19]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

Entrenamiento

In [ ]:
pipeline.fit(X, y)

C:\Users\Simon\Desktop\Simon\Programación en Python\swing_prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\Simon\Desktop\Simon\Programación en Python\swing_prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default v